# Más allá de Pearson y Spearman: dCor e Información Mutua

Pearson y Spearman son herramientas poderosas, pero tienen limitaciones concretas:  
- **Pearson** solo mide dependencia *lineal*.  
- **Spearman** extiende eso a relaciones *monótonas*, pero sigue siendo ciego a estructuras más complejas.

En este notebook mostramos casos donde ambos fallan, y presentamos dos alternativas más generales:
- **Distancia de Correlación (dCor)** — detecta cualquier tipo de dependencia, y dCor = 0 ↔ independencia.
- **Información Mutua (MI)** — mide cuánta información comparten dos variables, sin asumir ninguna forma funcional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

# dcor es una librería dedicada a distancia de correlación
# Instalación: pip install dcor
try:
    import dcor
    DCOR_OK = True
except ImportError:
    DCOR_OK = False
    print("Instalá dcor con: pip install dcor")

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("Librerías cargadas.")

---
## Función auxiliar

Calculamos las cuatro medidas para cada par (X, Y).

In [ ]:
def todas_las_medidas(x, y):
    """Devuelve dict con Pearson, Spearman, dCor e Información Mutua."""
    r,   _ = stats.pearsonr(x, y)
    rho, _ = stats.spearmanr(x, y)

    if DCOR_OK:
        dc = dcor.distance_correlation(x, y)
    else:
        dc = float('nan')

    # mutual_info_regression espera X como matriz columna
    mi = mutual_info_regression(x.reshape(-1, 1), y, random_state=0)[0]

    return {'Pearson r': r, 'Spearman ρ': rho, 'dCor': dc, 'MI (nats)': mi}


def scatter_4medidas(ax, x, y, titulo, color='steelblue'):
    """Scatter con las 4 medidas en el título."""
    m = todas_las_medidas(x, y)
    subtitle = (f"Pearson r={m['Pearson r']:.2f}  "
                f"Spearman ρ={m['Spearman ρ']:.2f}  "
                f"dCor={m['dCor']:.2f}  "
                f"MI={m['MI (nats)']:.2f}")
    ax.scatter(x, y, color=color, alpha=0.55, edgecolors='white', linewidths=0.3, s=45)
    ax.set_title(f"{titulo}\n{subtitle}", fontsize=9.5, pad=7)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    return m

---
## 1. Casos donde Pearson y Spearman fallan

Construimos cinco patrones con dependencia evidente pero que engañan a Pearson y/o Spearman.

In [ ]:
n = 300

# --- Patrones ---

# 1. Cuadrática simétrica: Y = X^2 + ruido
x1 = np.random.uniform(-3, 3, n)
y1 = x1**2 + np.random.normal(0, 0.6, n)

# 2. Circular
theta = np.random.uniform(0, 2 * np.pi, n)
x2 = np.cos(theta) + np.random.normal(0, 0.08, n)
y2 = np.sin(theta) + np.random.normal(0, 0.08, n)

# 3. Dos clusters simétricos (dependencia discreta)
cluster_a = np.random.multivariate_normal([-2, -2], [[0.2, 0], [0, 0.2]], n // 2)
cluster_b = np.random.multivariate_normal([ 2,  2], [[0.2, 0], [0, 0.2]], n // 2)
x3 = np.concatenate([cluster_a[:, 0], cluster_b[:, 0]])
y3 = np.concatenate([cluster_a[:, 1], cluster_b[:, 1]])
# Ahora agregamos un tercer cluster en la diagonal contraria para romper linealidad
cluster_c = np.random.multivariate_normal([-2, 2], [[0.2, 0], [0, 0.2]], n // 2)
cluster_d = np.random.multivariate_normal([ 2,-2], [[0.2, 0], [0, 0.2]], n // 2)
x3 = np.concatenate([x3, cluster_c[:, 0], cluster_d[:, 0]])
y3 = np.concatenate([y3, cluster_c[:, 1], cluster_d[:, 1]])

# 4. Zigzag / sinusoidal
x4 = np.random.uniform(0, 4 * np.pi, n)
y4 = np.sin(x4) + np.random.normal(0, 0.15, n)

# 5. Varianza heterogénea: Y ~ Normal(0, |X|)  — dependencia en dispersión
x5 = np.random.uniform(-4, 4, n)
y5 = np.random.normal(0, np.abs(x5) + 0.1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

scatter_4medidas(axes[0], x1, y1, 'Cuadrática  Y = X² + ruido',          color='#E91E63')
scatter_4medidas(axes[1], x2, y2, 'Patrón circular',                      color='#2196F3')
scatter_4medidas(axes[2], x3, y3, 'Cuatro clusters en X',                 color='#4CAF50')
scatter_4medidas(axes[3], x4, y4, 'Sinusoidal  Y = sin(X) + ruido',       color='#9C27B0')
scatter_4medidas(axes[4], x5, y5, 'Heteroscedasticidad  Var(Y) ∝ |X|',   color='#FF9800')
axes[5].axis('off')  # celda libre

fig.suptitle('Casos donde Pearson y Spearman se quedan cortos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observación:**  
En todos estos casos hay una relación clara visible en la nube de puntos, pero Pearson y/o Spearman reportan valores cercanos a cero.  
El patrón heteroscedástico es especialmente útil: la media de Y no depende de X, pero su *varianza* sí. Eso es dependencia real que ningún coeficiente clásico detecta.

En la fila de métricas notamos que **dCor** y **MI** toman valores claramente positivos donde Pearson y Spearman fallan.

---
## 2. Distancia de Correlación (dCor)

### ¿Qué es?

La **distancia de correlación** (*distance correlation*, dCor) fue propuesta por Székely, Rizzo y Bakirov (2007). A diferencia de Pearson, **dCor = 0 si y solo si X e Y son independientes** (para distribuciones con momentos finitos), independientemente de la forma de la relación.

### Idea intuitiva

En lugar de medir covarianza entre los valores de X e Y, mide covarianza entre las **matrices de distancias** de X e Y entre sí.  
Si saber la distancia entre dos observaciones de X nos dice algo sobre la distancia entre las correspondientes observaciones de Y, entonces hay dependencia.

### Fórmula (versión simplificada)

Dadas $n$ observaciones, definimos:

$$a_{kl} = \|X_k - X_l\|, \quad b_{kl} = \|Y_k - Y_l\|$$

Se centran estas matrices de distancias (doble centrado), obteniendo $A_{kl}$ y $B_{kl}$.  
Luego:

$$\text{dCov}^2(X, Y) = \frac{1}{n^2} \sum_{k,l} A_{kl} B_{kl}$$

$$\text{dCor}(X, Y) = \frac{\text{dCov}(X, Y)}{\sqrt{\text{dCov}(X, X) \cdot \text{dCov}(Y, Y)}}$$

### Propiedades clave

| Propiedad | Pearson | Spearman | **dCor** |
|---|---|---|---|
| Rango | [−1, 1] | [−1, 1] | **[0, 1]** |
| = 0 ↔ independencia | ✗ | ✗ | **✓** |
| Detecta no linealidad | ✗ | Parcial | **✓** |
| Tiene signo | ✓ | ✓ | **✗** |
| Interpretación | Lineal | Monótona | Cualquier dependencia |

> **Limitación:** dCor no tiene signo (siempre ≥ 0), por lo que no distingue entre correlación positiva y negativa. También es computacionalmente más costosa: O(n² log n).

In [ ]:
# Demostración: dCor en relaciones conocidas
n = 400
x = np.random.uniform(-3, 3, n)

casos = [
    ('Lineal positiva',          x,  2 * x + np.random.normal(0, 0.5, n)),
    ('Cuadrática  Y = X²',      x,  x**2 + np.random.normal(0, 0.5, n)),
    ('Cúbica  Y = X³',          x,  x**3 + np.random.normal(0, 2, n)),
    ('Independientes',           x,  np.random.normal(0, 1, n)),
]

colores = ['#2196F3', '#E91E63', '#9C27B0', '#607D8B']

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for i, (titulo, xi, yi) in enumerate(casos):
    scatter_4medidas(axes[i], xi, yi, titulo, color=colores[i])

fig.suptitle('dCor: detecta dependencia donde Pearson y Spearman fallan',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Información Mutua (MI)

### ¿Qué es?

La **información mutua** (*Mutual Information*, MI) proviene de la teoría de la información (Shannon, 1948).  
Mide cuánta incertidumbre sobre Y se reduce al conocer X, y viceversa.

### Idea intuitiva

Si X e Y son independientes, saber X no nos dice nada sobre Y → MI = 0.  
Si X determina completamente a Y, saber X elimina toda incertidumbre sobre Y → MI máxima.

### Fórmula

Para variables continuas:

$$I(X; Y) = \int\int p(x, y) \log \frac{p(x, y)}{p(x)\, p(y)}\, dx\, dy$$

donde $p(x,y)$ es la densidad conjunta y $p(x)$, $p(y)$ las marginales.  
En la práctica se estima con métodos de k-vecinos más cercanos (estimador de Kraskov).

### Propiedades clave

| Propiedad | Pearson | Spearman | dCor | **MI** |
|---|---|---|---|---|
| Rango | [−1, 1] | [−1, 1] | [0, 1] | **[0, ∞)** |
| = 0 ↔ independencia | ✗ | ✗ | ✓ | **✓** |
| Detecta no linealidad | ✗ | Parcial | ✓ | **✓** |
| Tiene signo | ✓ | ✓ | ✗ | **✗** |
| Escala interpretable | ✓ | ✓ | ✓ | **✗** (depende de la escala) |
| Invariante a transformaciones monótonas | ✗ | ✓ | ✗ | **✓** |

> **Limitaciones:** La MI no tiene una escala fija (valores en [0, ∞)), lo que dificulta comparar entre pares de variables de distinta naturaleza. Además, su estimación en muestras chicas puede ser inestable.  
> Para resolver la escala, existe el **MIC** (Maximal Information Coefficient) que normaliza MI a [0, 1].

In [ ]:
# Demostración: MI en relaciones conocidas
n = 400
x = np.random.uniform(-3, 3, n)

casos_mi = [
    ('Lineal positiva',               x,  2 * x + np.random.normal(0, 0.3, n),  '#2196F3'),
    ('Cuadrática  Y = X²',           x,  x**2 + np.random.normal(0, 0.4, n),    '#E91E63'),
    ('Sinusoidal  Y = sin(X)',        x,  np.sin(2*x) + np.random.normal(0, 0.2, n), '#4CAF50'),
    ('Heteroscedástica',              x,  np.random.normal(0, np.abs(x) + 0.1),  '#FF9800'),
    ('Independientes',                x,  np.random.normal(0, 1, n),             '#607D8B'),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
for i, (titulo, xi, yi, color) in enumerate(casos_mi):
    scatter_4medidas(axes[i], xi, yi, titulo, color=color)

fig.suptitle('Información Mutua: sensible a cualquier forma de dependencia',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Comparación directa: las cuatro medidas lado a lado

Construimos una tabla resumen para todos los patrones vistos.

In [ ]:
n = 400
x_base = np.random.uniform(-3, 3, n)
theta  = np.random.uniform(0, 2*np.pi, n)

patrones = {
    'Lineal':             (x_base, 2*x_base + np.random.normal(0, 0.4, n)),
    'Cuadrática':         (x_base, x_base**2 + np.random.normal(0, 0.5, n)),
    'Cúbica':             (x_base, x_base**3 + np.random.normal(0, 3, n)),
    'Sinusoidal':         (x_base, np.sin(2*x_base) + np.random.normal(0, 0.2, n)),
    'Circular':           (np.cos(theta) + np.random.normal(0, 0.08, n),
                           np.sin(theta) + np.random.normal(0, 0.08, n)),
    'Heteroscedástica':   (x_base, np.random.normal(0, np.abs(x_base)+0.1)),
    'Independientes':     (x_base, np.random.normal(0, 1, n)),
}

print(f"{'Patrón':<22} {'Pearson r':>10} {'Spearman ρ':>12} {'dCor':>8} {'MI (nats)':>11}")
print("-" * 67)

for nombre, (xi, yi) in patrones.items():
    m = todas_las_medidas(xi, yi)
    print(f"{nombre:<22} {m['Pearson r']:>10.3f} {m['Spearman ρ']:>12.3f} "
          f"{m['dCor']:>8.3f} {m['MI (nats)']:>11.3f}")

**Lectura de la tabla:**
- Pearson y Spearman son ≈ 0 en cuadrática, circular, heteroscedástica → falsa sensación de independencia.
- dCor y MI reportan valores positivos en todos los casos con dependencia real.
- En el caso de variables **genuinamente independientes**, los cuatro valores son ≈ 0.

> **Conclusión práctica:** Pearson y Spearman son suficientes para exploración rápida cuando se espera linealidad o monotonicidad. Para exploración general sin supuestos de forma, dCor o MI son más seguros. En aprendizaje automático, MI se usa mucho en selección de features.

---
## 5. Celda de exploración libre

Definí tu propia relación funcional y observá cómo responden las cuatro medidas.

In [ ]:
# ============================================================
#  DEFINÍ TU PROPIA RELACIÓN
# ============================================================
n   = 300
x   = np.random.uniform(-3, 3, n)

# Modificá esta línea con la relación que quieras explorar:
y   = np.abs(x) + np.random.normal(0, 0.4, n)   # ← cambiá esto

# Ejemplos para probar:
#   y = x**3
#   y = np.sign(x) * x**2 + np.random.normal(0, 0.3, n)
#   y = np.exp(x) + np.random.normal(0, 0.5, n)
#   y = (x > 0).astype(float) + np.random.normal(0, 0.2, n)   # escalón
#   y = np.random.normal(0, 1, n)                              # independientes
# ============================================================

m = todas_las_medidas(x, y)

fig, ax = plt.subplots(figsize=(6, 5))
scatter_4medidas(ax, x, y, 'Tu relación personalizada', color='#00BCD4')
plt.tight_layout()
plt.show()

print("\nResumen de medidas:")
for k, v in m.items():
    print(f"  {k:<15}: {v:.4f}")